# 08 · M1.4 — M1.3 with one Student-t ν per family, the best on the backtest

## Executive summary (read this first)

**M1.4 = M1.3 with one change:** the Student-t's ν is not fitted from each date's window; each family (F1, F2, F3, F4)
gets **one global ν**, the value that gives the family's best mean competition score on the backtest. Everything else is
M1.3: the transform per asset type, the drift rule (drift for CPI and NFP only), the fixed 260-step window for the drift,
the sd and the correlation, and the Student-t only on rates, FX and equity factors (CPI, NFP, UNRATE stay normal).

The candidates: ν ∈ {3, 4, 5, 6, 8, 10, 15, 30} and normal (no Student-t). The Student-t keeps the variance (each draw's
shock is multiplied by `√((ν − 2) ÷ W)`, `W ~ χ²(ν)`, one per draw): a smaller ν means a narrower middle and fatter tails.

**The same backtest dates both choose ν and score M1.4, so its number is optimistic** (an upper bound for what a per-family
ν can give). One number per family is chosen, so the optimism is small.

Results: §2 (the ν grid per family), §3 (every model side by side), §4 (per family and asset type), §6 (in short). No
realized outcome is read.

## 0 — Setup (as notebook 07)

In [1]:
import bisect, multiprocessing as mp, pathlib, sys, time, tomllib, warnings, zlib
import numpy as np, pandas as pd, pyarrow.parquet as pq

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "model_experiment_v3" else pathlib.Path.cwd() / "model_experiment_v3"
REPO, UNITS, DATA = HERE.parent, HERE.parent / "units", HERE / "data"
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
import m0_baseline as m0
from qfbench2_track_forecasting import scoring as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))

ANSWERS = pd.read_parquet(DATA / "backtest_answers.parquet")
M0_SCORES = pd.read_parquet(DATA / "m0_backtest_scores.parquet")
M11_SCORES = pd.read_parquet(DATA / "m1_1_backtest_scores.parquet")
UNIT_IDS = sorted(ANSWERS.unit.unique())
RULES = tomllib.loads((HERE / "transformations.toml").read_text())
SH = RULES["shocks"]
WINDOW = 260

def asset_type(a):
    return RULES["assets"].get(a, "other")

def rule_for(asset, target_type):
    """card.toml's target type + the asset -> {transform, shock, drift} (transformations.toml)."""
    t = RULES["assets"].get(asset)
    base = RULES["types"][t] if t else {"transform": RULES["default"]["level"], "shock": RULES["default"]["shock"],
                                        "drift": RULES["default"]["drift"]}
    tr = RULES["default"]["log_return"] if target_type == "log_return" else base["transform"]
    return {"transform": tr, "shock": base["shock"], "drift": bool(base["drift"])}

def load_unit(u):
    ud = UNITS / u
    card = tomllib.loads((ud / "card.toml").read_text()); t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    uid, asof, tt = card["task"]["id"], card["provenance"]["data_cutoff"], t.get("target_type", "level")
    rows = {}
    for a in assets:
        for path in sorted(ud.glob("*.parquet")):
            tab = pq.read_table(path).to_pydict()
            acol = "asset" if "asset" in tab else ("asset_id" if "asset_id" in tab else None)
            if acol is None:
                continue
            r = sorted((str(d)[:10], float(v)) for d, x, v in zip(tab["date"], tab[acol], tab["value"])
                       if str(x) == a and str(d)[:10] <= asof and v is not None)
            if r:
                rows[a] = r; break
    return {"unit": u, "id": uid, "family": card["metadata"]["category"], "card": card, "assets": assets,
            "horizons": horizons, "target_type": tt, "asof": asof, "rows": rows, "dates": {a: [d for d, _ in rows[a]] for a in assets},
            "rules": {a: rule_for(a, tt) for a in assets}, "steps": {h: m0._MONTHLY_STEPS.get(uid, {}).get(h, h) for h in horizons}}

UNITS_DATA = {u: load_unit(u) for u in UNIT_IDS}
TYPE = {u: "+".join(sorted({asset_type(a) for a in U["assets"]})) for u, U in UNITS_DATA.items()}
BACKTEST = {}
for u, g in ANSWERS.groupby("unit"):
    U = UNITS_DATA[u]
    order = {(a, h): i for i, (a, h) in enumerate((a, h) for a in U["assets"] for h in U["horizons"])}
    BACKTEST[u] = [(o, gg.assign(i=[order[(a, h)] for a, h in zip(gg.asset, gg.horizon)]).sort_values("i").answer.to_numpy())
                   for o, gg in g.groupby("origin")]
print(f"{len(UNIT_IDS)} units, {sum(len(v) for v in BACKTEST.values()):,} backtest dates")
display(pd.DataFrame({t: {"transform": v["transform"], "shock": v["shock"], "drift": v["drift"]} for t, v in RULES["types"].items()}).T)
print("Student-t:", SH)

103 units, 23,635 backtest dates


,transform,shock,drift
rates,diff,student_t,False
fx,log_diff,student_t,False
cpi,log_diff,normal,True
payrolls,diff,normal,True
unemployment,diff,normal,False
equity_factor,as_return,student_t,False


Student-t: {'nu': 'fit', 'nu_min': 4.2, 'nu_max': 30.0, 'normal_below_kurtosis': 0.2}


## 1 — The model: notebook 07's M1.3 with an optional fixed ν

`m13_draws(U, origin, nu=...)`: `nu=None` is M1.3 (ν fitted from the window), a number is that ν for every date (M1.4),
`"normal"` switches the Student-t off. Check: `nu=None` reproduces notebook 07's M1.3 scores.

In [2]:
def fit_nu(x, rows_t):
    """nu from the window's excess kurtosis k: a Student-t has k = 6 / (nu - 4), so nu = 4 + 6 / k."""
    if not rows_t:
        return None
    z = x[rows_t] - x[rows_t].mean(axis=1, keepdims=True)
    k = float(np.mean((z ** 4).mean(axis=1) / (z ** 2).mean(axis=1) ** 2 - 3))
    return None if k < SH["normal_below_kurtosis"] else float(np.clip(4 + 6 / k, SH["nu_min"], SH["nu_max"]))

def fit(U, origin, steps_window=None):
    """The window's steps per asset -> (assets sorted, steps matrix x, log mode, last value)."""
    by_asset = sorted(U["assets"]); steps, log_mode, last = {}, {}, {}
    for a in by_asset:
        k = bisect.bisect_right(U["dates"][a], origin)
        rows = U["rows"][a][max(0, k - (m0._WINDOW if steps_window is None else steps_window + 1)): k]
        log_mode[a] = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
        if log_mode[a]:
            steps[a], _ = m0._steps([(d, np.log(v)) for d, v in rows], "level"); last[a] = rows[-1][1]
        else:
            steps[a], lv = m0._steps(rows, U["target_type"]); last[a] = lv if U["target_type"] == "level" else 0.0
    common = sorted(set.intersection(*(set(steps[a]) for a in by_asset)))
    x = np.array([[steps[a][d] for d in common] for a in by_asset])
    return by_asset, x, log_mode, last

def m13_draws(U, origin, steps_window=WINDOW, drift_rule=True, student_t=True, nu=None):
    """nu=None with student_t: nu fitted from the window (M1.3); a number: that fixed nu (M1.4); "normal": no t."""
    by_asset, x, log_mode, last = fit(U, origin, steps_window)
    assets, horizons = U["assets"], U["horizons"]
    mu, sigma = x.mean(axis=1), np.atleast_2d(np.cov(x))
    ai = {a: i for i, a in enumerate(by_asset)}
    cells = [(a, h) for a in by_asset for h in sorted(horizons)]; s = U["steps"]
    def centre(a, h):
        anchor = 0.0 if log_mode[a] else last[a]
        use_drift = U["rules"][a]["drift"] if drift_rule else True
        return anchor + (s[h] * mu[ai[a]] if use_drift else 0.0)
    mean = np.array([centre(a, h) for a, h in cells])
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10; j = cov + 1e-9 * np.eye(len(cells))
    try: f = np.linalg.cholesky(j)
    except np.linalg.LinAlgError: f = np.diag(np.sqrt(np.diag(j)))
    seed = zlib.crc32(U["id"].encode()) & 0x7FFFFFFF
    dev = np.random.default_rng(seed).standard_normal((m0.N_DRAWS, len(cells))) @ f.T
    t_cells = np.array([U["rules"][a]["shock"] == "student_t" for a, h in cells])
    if not student_t or nu == "normal":
        nu = None
    elif nu is None:
        nu = fit_nu(x, [ai[a] for a in by_asset if U["rules"][a]["shock"] == "student_t"])
    if nu is not None and t_cells.any():
        dev = dev * np.where(t_cells, np.sqrt((nu - 2.0) / np.random.default_rng([seed, 7]).chisquare(nu, size=(m0.N_DRAWS, 1))), 1.0)
    sm = mean + dev
    out = np.empty((m0.N_DRAWS, len(assets), len(horizons)))
    for ci, (a, h) in enumerate(cells):
        col = last[a] * np.exp(sm[:, ci]) if log_mode[a] else sm[:, ci]
        if s[h] <= 0:
            col = np.full(m0.N_DRAWS, U["rows"][a][bisect.bisect_right(U["dates"][a], origin) - 1][1])
        out[:, assets.index(a), horizons.index(h)] = col
    return out


cats = ["marginal", "joint", "tail"]

def parts(card, samples, y):
    p = card.get("scoring", {}).get("params", {})
    return S._composite(samples, y, weights=m0.card_weights(card, samples.shape[1]),
                        tail_levels=tuple(p.get("tail_levels", (0.01, 0.05, 0.95, 0.99))), joint=S.card_joint_statistic(card),
                        tail_metric=str(p.get("tail_metric", S.DEFAULT_TAIL_METRIC)), ref_scale=None)
mean0 = M0_SCORES.groupby("unit")[cats].mean()
def weights(u):
    U = UNITS_DATA[u]; return m0.card_weights(U["card"], len(U["assets"]) * len(U["horizons"]))
def vs_m0(scores):
    m = scores.groupby("unit")[cats].mean()
    return pd.Series({u: sum(wk * m.loc[u, k] / mean0.loc[u, k] for k, wk in zip(cats, weights(u)) if wk) for u in UNIT_IDS})

NUS = [3, 4, 5, 6, 8, 10, 15, 30, "normal", None]          # None = fitted (M1.3), for the check and the reference
def score_unit(u):
    U = UNITS_DATA[u]
    return [{"unit": u, "family": U["family"], "origin": o, "nu": str(nu), **parts(U["card"], m13_draws(U, o, nu=nu).reshape(500, -1), y)}
            for o, y in BACKTEST[u] for nu in NUS]
t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    G = pd.DataFrame([r for rows in pool.map(score_unit, UNIT_IDS, chunksize=1) for r in rows])
print(f"scored {len(G) // len(NUS):,} dates x {len(NUS)} nu settings in {time.time() - t0:.0f}s")
m13_saved = vs_m0(pd.read_parquet(DATA / "m1_3_backtest_scores.parquet"))
d = float((vs_m0(G[G.nu == "None"]) - m13_saved).abs().max())
print(f"nu fitted (None) vs notebook 07's saved M1.3, competition score per unit: max |difference| = {d:.1e}")
assert d < 1e-12

scored 23,635 dates x 10 nu settings in 13s
nu fitted (None) vs notebook 07's saved M1.3, competition score per unit: max |difference| = 0.0e+00


## 2 — The ν grid per family, and the best ν

Mean competition score vs M0 over the family's units, for each ν (rows) and family (columns). The lowest per column is the
family's ν.

In [3]:
fam = pd.Series({u: UNITS_DATA[u]["family"] for u in UNIT_IDS})
SCORE = {nu: vs_m0(G[G.nu == nu]) for nu in G.nu.unique()}
order = [str(n) for n in NUS if n is not None]
GRID = pd.DataFrame({f: {nu: SCORE[nu][fam[fam == f].index].mean() for nu in order} for f in sorted(fam.unique())})
GRID["all units"] = [SCORE[nu].mean() for nu in order]
GRID.loc["fitted per date (M1.3)"] = [SCORE["None"][fam[fam == f].index].mean() for f in sorted(fam.unique())] + [SCORE["None"].mean()]
display(GRID)
BEST = {f: GRID.loc[order, f].idxmin() for f in sorted(fam.unique())}
print("best nu per family (on the backtest):", BEST)
M14 = pd.concat([G[(G.nu == BEST[f]) & (G.family == f)] for f in BEST]).drop(columns="nu").sort_values(["unit", "origin"]).reset_index(drop=True)
assert len(M14) == len(M0_SCORES)

,T2-F1,T2-F2,T2-F3,T2-F4,all units
3,0.8947,1.01,0.9389,0.9941,0.9644
4,0.8929,1.001,0.9245,0.9818,0.9546
5,0.8945,0.9996,0.9208,0.9769,0.9524
6,0.8944,0.999,0.9191,0.9771,0.952
8,0.895,1.002,0.9201,0.9776,0.9533
10,0.8961,1.002,0.9197,0.9771,0.9532
15,0.8978,0.9999,0.9203,0.9771,0.9532
30,0.9003,1.001,0.921,0.9781,0.9546
normal,0.9014,1.004,0.9227,0.9803,0.9566
fitted per date (M1.3),0.8951,0.9995,0.9194,0.9763,0.9521


best nu per family (on the backtest): {'T2-F1': '4', 'T2-F2': '6', 'T2-F3': '6', 'T2-F4': '5'}


## 3 — Every model on the same backtest

In [4]:
_w12 = pd.read_parquet(DATA / "m1_2_window_backtest_scores.parquet")
models = {"M0": M0_SCORES, "M1.1 (per asset type)": M11_SCORES,
          "M1.2 (t fitted, 2 years, M0 drift)": _w12[_w12.window == 504].drop(columns="window"),
          "production (text off)": pd.read_parquet(DATA / "production_backtest_scores.parquet"),
          "M1.3 (nu fitted per date)": pd.read_parquet(DATA / "m1_3_backtest_scores.parquet"),
          "M1.4 (best nu per family)": M14}
SC = {name: vs_m0(sc) for name, sc in models.items()}
LADDER = pd.DataFrame({name: {**{f: s[fam[fam == f].index].mean() for f in sorted(fam.unique())}, "all": s.mean(),
                              "better than M0": f"{(s < 1).sum()}/{len(s)}"} for name, s in SC.items()}).T
LADDER

,T2-F1,T2-F2,T2-F3,T2-F4,all,better than M0
M0,1,1,1,1,1,0/103
M1.1 (per asset type),0.9955,0.999,0.9993,0.9968,0.9976,25/103
"M1.2 (t fitted, 2 years, M0 drift)",0.97,0.9967,0.9761,1.01,0.9902,64/103
production (text off),0.9288,1.011,0.9274,1.003,0.9726,74/103
M1.3 (nu fitted per date),0.8951,0.9995,0.9194,0.9763,0.9521,81/103
M1.4 (best nu per family),0.8929,0.999,0.9191,0.9769,0.9516,81/103


## 4 — Per family and per asset type: M1.3 vs M1.4

In [5]:
PER_UNIT = pd.DataFrame({"family": fam, "asset type": pd.Series(TYPE), "M1.3 vs M0": SC["M1.3 (nu fitted per date)"],
                         "M1.4 vs M0": SC["M1.4 (best nu per family)"], "production vs M0": SC["production (text off)"]})
PER_UNIT["nu used (M1.4)"] = PER_UNIT.family.map(BEST)
def summary(g):
    return {"units": len(g), "M1.3": g["M1.3 vs M0"].mean(), "M1.4": g["M1.4 vs M0"].mean(), "production": g["production vs M0"].mean(),
            "M1.4 better than M1.3": f'{(g["M1.4 vs M0"] < g["M1.3 vs M0"]).sum()}/{len(g)}',
            "M1.4 better than production": f'{(g["M1.4 vs M0"] < g["production vs M0"]).sum()}/{len(g)}'}
FAMILY = pd.DataFrame({f: summary(g) for f, g in [*PER_UNIT.groupby("family"), ("all", PER_UNIT)]}).T.rename_axis("family")
BY_TYPE = pd.DataFrame({t: summary(g) for t, g in PER_UNIT.groupby("asset type")}).T.rename_axis("asset type")
print("per family:"); display(FAMILY.assign(**{"nu (M1.4)": pd.Series(BEST)}))
print("per asset type:"); display(BY_TYPE)
print("per unit:"); display(PER_UNIT.rename_axis("unit").reset_index().set_index(["family", "unit"]).sort_index())

per family:


,units,M1.3,M1.4,production,M1.4 better than M1.3,M1.4 better than production,nu (M1.4)
family,,,,,,,
T2-F1,23,0.8951,0.8929,0.9288,9/23,13/23,4
T2-F2,27,0.9995,0.999,1.011,13/27,15/27,6
T2-F3,22,0.9194,0.9191,0.9274,13/22,13/22,6
T2-F4,31,0.9763,0.9769,1.003,14/31,18/31,5
all,103,0.9521,0.9516,0.9726,49/103,59/103,NaN


per asset type:


,units,M1.3,M1.4,production,M1.4 better than M1.3,M1.4 better than production
asset type,,,,,,
cpi,2,0.8735,0.8735,1.308,0/2,2/2
equity_factor,16,0.9758,0.9765,0.9935,8/16,10/16
fx,35,0.9685,0.968,0.9819,20/35,21/35
fx+rates,5,0.9465,0.9469,0.9429,1/5,2/5
payrolls,1,1,1,1.624,0/1,1/1
rates,43,0.9323,0.9311,0.9285,20/43,22/43
unemployment,1,0.9875,0.9875,1.036,0/1,1/1


per unit:


asset type  M1.3 vs M0  M1.4 vs M0  production vs M0 nu used (M1.4)
family unit                                                                                                     
T2-F1  t2-F1-ai-mom-2024                  equity_factor      0.7878      0.7821              0.77              4
       t2-F1-aud-on-hold-2016                        fx      0.9182      0.9168            0.9478              4
       t2-F1-cad-boc-2017                            fx      0.9815      0.9903             1.037              4
       t2-F1-chf-highly-valued-2021                  fx      0.8872      0.8652            0.8065              4
       t2-F1-conflicting-texts-2024               rates      0.9962       1.026             1.026              4
       t2-F1-considerable-period-2003             rates      0.9263      0.9341            0.9141              4
       t2-F1-conundrum-2005                       rates      0.7077      0.6669            0.6256              4
       t2-F1-cpi-glidepath-2023                     cpi      0.8531      0.8531             1.434              4
       t2-F1-dkk-peg-2019                            fx      0.7668      0.7639            0.8197              4
       t2-F1-eur-range-2017                          fx      0.7981      0.7713            0.8291              4
       t2-F1-fed-put-2019                 equity_factor       1.116       1.125             1.196              4
       t2-F1-greater-confidence-2024              rates      0.9441      0.9534            0.9257              4
       t2-F1-hawkish-cut-2024                     rates      0.9135      0.9287            0.8902              4
       t2-F1-jpy-ycc-flex-2018                       fx      0.8435      0.8667            0.9337              4
       t2-F1-liftoff-telegraph-2015               rates      0.9131      0.8588            0.7836              4
       t2-F1-measured-pace-2004                   rates      0.6306      0.6033            0.5517              4
       t2-F1-normalization-2017                   rates      0.7657      0.7753             0.826              4
       t2-F1-patient-dropped-2015                 rates      0.9078      0.8666             0.786              4
       t2-F1-pause-2006                           rates       1.298       1.312             1.372              4
       t2-F1-qe3-end-2014                         rates      0.8192      0.8245            0.8437              4
       t2-F1-sahm-watch-2024               unemployment      0.9875      0.9875             1.036              4
       t2-F1-third-cut-2019                       rates       1.004       1.037             1.116              4
       t2-F1-ust-positioning-2018                 rates      0.8208      0.8274            0.8929              4
T2-F2  t2-F2-abenomics-2012                          fx      0.9095      0.9091             0.881              6
       t2-F2-act-as-appropriate-2019              rates      0.9352       0.941             1.024              6
       t2-F2-aud-taper-2013                          fx      0.9109       0.909            0.8554              6
       t2-F2-boj-thin-signal-2014                    fx       1.013       1.021              1.05              6
       t2-F2-cnh-stress-2015                         fx       0.988      0.9913            0.9853              6
       t2-F2-cnh-tradewar-2018                       fx       2.063       2.072             1.842              6
       t2-F2-considerable-removal-2004            rates      0.7335      0.7205            0.7274              6
       t2-F2-credit-crunch-2007                   rates       1.011       1.024             1.053              6
       t2-F2-cut-sizing-2024                      rates      0.9789      0.9766            0.9503              6
       t2-F2-draghi-precommit-2014                   fx      0.8315      0.8344            0.8414              6
       t2-F2-ecb-qe-telegraph-2014                   fx      0.8514      0.8465           

## 5 — Save

In [6]:
M14.to_parquet(DATA / "m1_4_backtest_scores.parquet", index=False)
GRID.to_csv(DATA / "m1_4_nu_grid.csv"); PER_UNIT.to_csv(DATA / "m1_4_vs_m0_per_unit.csv")
FAMILY.to_csv(DATA / "m1_4_vs_m0_per_family.csv"); BY_TYPE.to_csv(DATA / "m1_4_vs_m0_by_asset_type.csv")
for f in ("m1_4_backtest_scores.parquet", "m1_4_nu_grid.csv", "m1_4_vs_m0_per_unit.csv", "m1_4_vs_m0_per_family.csv", "m1_4_vs_m0_by_asset_type.csv"):
    print(f"data/{f:<30} {(DATA / f).stat().st_size / 1e3:8.1f} kB")

data/m1_4_backtest_scores.parquet      867.0 kB
data/m1_4_nu_grid.csv                    1.0 kB
data/m1_4_vs_m0_per_unit.csv             9.9 kB
data/m1_4_vs_m0_per_family.csv           0.5 kB
data/m1_4_vs_m0_by_asset_type.csv        0.6 kB


## 6 — In short

In [7]:
print("best nu per family (chosen on the backtest, so M1.4 is optimistic):", BEST)
for name in LADDER.index:
    print(f"  {name:<36} {LADDER.loc[name, 'all']:.4f}   (better than M0 on {LADDER.loc[name, 'better than M0']})")
a = FAMILY.loc["all"]
print(f"M1.4 vs M1.3: better on {a['M1.4 better than M1.3']} units; vs production: better on {a['M1.4 better than production']}")

best nu per family (chosen on the backtest, so M1.4 is optimistic): {'T2-F1': '4', 'T2-F2': '6', 'T2-F3': '6', 'T2-F4': '5'}
  M0                                   1.0000   (better than M0 on 0/103)
  M1.1 (per asset type)                0.9976   (better than M0 on 25/103)
  M1.2 (t fitted, 2 years, M0 drift)   0.9902   (better than M0 on 64/103)
  production (text off)                0.9726   (better than M0 on 74/103)
  M1.3 (nu fitted per date)            0.9521   (better than M0 on 81/103)
  M1.4 (best nu per family)            0.9516   (better than M0 on 81/103)
M1.4 vs M1.3: better on 49/103 units; vs production: better on 59/103
